<a href="https://colab.research.google.com/github/SangwanJeon/sensex-corpus-workflow-public/blob/main/01_data_collection/pipelines/youtube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U openai-whisper
!pip install pympi-ling
!pip install -U yt-dlp
!apt-get update
!apt-get install -y ffmpeg


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,933 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [ ]:
import os
import subprocess
import whisper
import pympi

VIDEO_NAME = "LegendMedia" #INSERT VIDEO NAME
YOUTUBE_URL = "https://www.youtube.com/watch?v=sFgm6y4uqSQ" #INSERT YOUR VIDEO URL

SEGMENTS = [
    ("excessive_legendmedia", 30, 84), #INSERT SEGMENT_1 NAME, BEGIN TIME (IN SECONDS), END TIME (IN SECONDS)
    ("not_excessive_legendmedia", 739, 793),  #INSERT SEGMENT_2 NAME, BEGIN TIME (IN SECONDS), END TIME (IN SECONDS)
]

os.makedirs("segments", exist_ok=True)
os.makedirs("video_segments", exist_ok=True)
os.makedirs("eaf_segments", exist_ok=True)

In [ ]:
# 1. Download entire video
video_file = f"{VIDEO_NAME}.mp4"
subprocess.run([
    "yt-dlp",
    "-f", "mp4",
    "-o", video_file,
    YOUTUBE_URL
], check=True)

CompletedProcess(args=['yt-dlp', '-f', 'mp4', '-o', 'LegendMedia.mp4', 'https://www.youtube.com/watch?v=sFgm6y4uqSQ'], returncode=0)

In [ ]:
# 2. Extract audio in .wav
audio_file = f"{VIDEO_NAME}.wav"
subprocess.run([
    "ffmpeg",
    "-y",
    "-i", video_file,
    "-ar", "16000",
    "-ac", "1",
    audio_file
], check=True)


CompletedProcess(args=['ffmpeg', '-y', '-i', 'LegendMedia.mp4', '-ar', '16000', '-ac', '1', 'LegendMedia.wav'], returncode=0)

In [ ]:
# 3. Call Whisper model
model = whisper.load_model("base")




100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 295MiB/s]


In [ ]:
# 4. Process the segments
for name, start, end in SEGMENTS:
    segment_audio = f"segments/{name}.wav"
    segment_video = f"video_segments/{name}.mp4"
    segment_eaf_file = f"eaf_segments/{name}.eaf"

    # Cut audio
    subprocess.run([
        "ffmpeg",
        "-y",
        "-i", audio_file,
        "-ss", str(start),
        "-to", str(end),
        "-ar", "16000",
        "-ac", "1",
        segment_audio
    ], check=True)

    # Cut video
    subprocess.run([
        "ffmpeg",
        "-y",
        "-i", video_file,
        "-ss", str(start),
        "-to", str(end),
        "-c", "copy",
        segment_video
    ], check=True)

    # Create eaf file
    eaf = pympi.Elan.Eaf()
    eaf.add_linked_file(segment_video)
    tier_name = "transcription"
    eaf.add_tier(tier_name)

    # Transcribe Audio
    result = model.transcribe(segment_audio)

    for seg in result["segments"]:
        start_ms = int(seg["start"] * 1000)
        end_ms = int(seg["end"] * 1000)
        text = seg["text"].strip()
        eaf.add_annotation(tier_name, start_ms, end_ms, text)

    # Save eaf file
    eaf.to_file(segment_eaf_file)
    print(f"✅ Segmento '{name}' completato: {segment_video}, {segment_eaf_file}")


✅ Segmento 'excessive_legendmedia' completato: video_segments/excessive_legendmedia.mp4, eaf_segments/excessive_legendmedia.eaf
✅ Segmento 'not_excessive_legendmedia' completato: video_segments/not_excessive_legendmedia.mp4, eaf_segments/not_excessive_legendmedia.eaf


In [ ]:
print("🎉 Completed!")

🎉 Completed!
